# Workout Tracker: Learning Journal

A documented build log covering Docker, PostgreSQL, relational schema design, Python, and Streamlit — including the reasoning behind each decision, not just the final code.

**Stack:** Docker · PostgreSQL · Python (psycopg2) · Streamlit

## 1. Docker Fundamentals

**The core idea:** instead of installing software directly on a machine (which risks version conflicts and "works on my machine" problems), Docker packages an application together with everything it needs into a portable **container**. A container is built from an **image** — a downloadable blueprint (from Docker Hub) that defines what's inside.

**Key commands learned:**

| Command | Purpose |
|---|---|
| `docker run <image>` | Create and start a new container from an image |
| `docker ps` | List currently running containers |
| `docker ps -a` | List all containers, including stopped ones |
| `docker images` | List downloaded images (blueprints) on this machine |
| `docker stop <name>` | Stop a running container without deleting it |
| `docker start <name>` | Restart a previously stopped container |
| `docker rm <id>` | Permanently delete a container |
| `docker rmi <image>` | Permanently delete a downloaded image |
| `docker exec -it <name> <cmd>` | Run a command inside an already-running container |

**Important distinction:** removing a *container* does not remove the *image* it came from — the image (blueprint) stays, so new containers can be created from it instantly without re-downloading.

## 2. Running Postgres in Docker

```
docker run --name workout-db -e POSTGRES_PASSWORD=<password> -p 5432:5432 -d postgres
```

- `--name workout-db` — friendly name instead of a random one
- `-e POSTGRES_PASSWORD=...` — sets a required environment variable inside the container
- `-p 5432:5432` — maps port 5432 on the host machine to port 5432 inside the container (Postgres's default port), so tools outside the container can reach it
- `-d` — runs the container in the background ("detached")

**Note on the password above:** in this notebook, the real password is never written directly — it's loaded from a `.env` file (see Section 3) and referenced only as `DB_PASSWORD`. The `.env` file itself is excluded from version control via `.gitignore`, so it never gets pushed to GitHub.

## 3. Connecting from Python (with secrets handled properly)

Credentials are stored in a local `.env` file (never committed to Git) and loaded with `python-dotenv`, rather than hardcoded in the script.

**.env file (local only, git-ignored):**
```
DB_PASSWORD=<actual password lives only here, on this machine>
```

**.gitignore:**
```
.env
__pycache__/
```

In [ ]:
import os
from dotenv import load_dotenv
import psycopg2

load_dotenv()  # reads the local .env file into environment variables

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="postgres",
    user="postgres",
    password=os.getenv("DB_PASSWORD")  # never hardcoded
)
cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone())

**The connection pattern, explained:**

- `conn` = the open connection itself (like an open phone line)
- `cur` (cursor) = the tool used to actually send SQL and receive results over that connection
- `cur.execute(sql, params)` = send a SQL statement (optionally with `%s` placeholders filled in safely from `params`, avoiding SQL injection)
- `conn.commit()` = permanently save any changes (`INSERT`/`UPDATE`/`DELETE`) — without this, changes are discarded
- `cur.close()` / `conn.close()` = properly end the session when done

## 4. Relational Schema Design

Started with a single flat table, then redesigned into a normalized relational schema after recognizing repeated text values (exercise names, workout types) needed to become reference tables instead.

**Core design principles learned:**

1. **Reference tables** hold rarely-changing lookup values (e.g. `types`, `muscles`, `equipment`, `categories`) — similar to an Excel dropdown list, but enforced by the database instead of just suggested.
2. **One-to-many relationships** (e.g. one session has many workout logs) are represented with a simple foreign key column: `session_id INTEGER REFERENCES sessions(id)`.
3. **Many-to-many relationships** (e.g. one exercise can have multiple muscles, and one muscle appears in multiple exercises) cannot be represented with a single column — they require a separate **bridge table** holding pairs of IDs.
4. **Avoid redundancy:** a fact should be stored in exactly one place. Example: rest time was moved from `exercises` to `categories`, since it was actually determined by category (Compound/Isolation/etc.), not by the individual exercise — storing it in both places would risk contradictory data.

**Final tables:**

| Table | Purpose | Relationship type |
|---|---|---|
| `types` | Workout day types (Push, Pull, Legs, Upper, Lower, Full Body) | Reference table |
| `categories` | Movement categories (Compound, Isolation, Cardio, Isometric) with `rest_seconds` | Reference table |
| `exercises` | One row per exercise, referencing `categories` | One-to-many (category → exercises) |
| `exercise_types` | Bridge: exercise ↔ day-type | Many-to-many |
| `muscles` | Muscle groups | Reference table |
| `exercise_muscles` | Bridge: exercise ↔ muscle | Many-to-many |
| `equipment` | Gym equipment | Reference table |
| `exercise_equipment` | Bridge: exercise ↔ equipment | Many-to-many |
| `sessions` | One row per real workout day, referencing `types` | One-to-many (type → sessions) |
| `workout_logs` | One row per logged set, referencing `sessions` and `exercises` | Many-to-one (both directions) |

In [ ]:
cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")
for row in cur.fetchall():
    print(row[0])

## 5. Populating Data

Reference tables (types, categories, muscles, equipment) were populated first, since other tables depend on them. Then 53 real exercises were inserted, each linked to its category, day-type(s), muscle(s), and equipment via subqueries matching on name:

```sql
INSERT INTO exercise_types (exercise_id, type_id)
SELECT e.id, t.id FROM exercises e, types t
WHERE e.name = %s AND t.name = %s;
```

Rather than writing this by hand 53+ times, the data was structured as a Python list of dictionaries and inserted via a loop — a "don't repeat yourself" pattern that scales far better than hardcoded SQL blocks.

In [ ]:
for label, table in [("Exercises", "exercises"), ("Exercise-Type links", "exercise_types"),
                      ("Exercise-Muscle links", "exercise_muscles"), ("Exercise-Equipment links", "exercise_equipment")]:
    cur.execute(f"SELECT COUNT(*) FROM {table};")
    print(f"{label}: {cur.fetchone()[0]}")

## 6. SQL: JOINs

A `JOIN` combines rows from two tables based on a matching condition — this is what turns raw foreign key numbers back into readable names.

```sql
SELECT e.name AS exercise, c.name AS category, c.rest_seconds
FROM exercises e
JOIN categories c ON e.category_id = c.id;
```

Reading it: "start with exercises, also bring in categories, but only where the exercise's `category_id` matches the category's `id`."

In [ ]:
cur.execute("""
    SELECT e.name AS exercise, c.name AS category, c.rest_seconds
    FROM exercises e
    JOIN categories c ON e.category_id = c.id
    LIMIT 5;
""")
for row in cur.fetchall():
    print(row)

## 7. SQL: EXCEPT (set difference)

`EXCEPT` returns rows from the first query that do **not** appear in the second query's results — used here to answer "which muscles are reachable for this workout type, but haven't been logged yet this session?"

```sql
SELECT DISTINCT mu.name FROM exercises e
JOIN exercise_types et ON e.id = et.exercise_id
JOIN types t ON et.type_id = t.id
JOIN exercise_muscles em ON e.id = em.exercise_id
JOIN muscles mu ON em.muscle_id = mu.id
WHERE t.name = %s

EXCEPT

SELECT DISTINCT mu.name FROM workout_logs wl
JOIN exercise_muscles em ON wl.exercise_id = em.exercise_id
JOIN muscles mu ON em.muscle_id = mu.id
WHERE wl.session_id = %s;
```

`DISTINCT` prevents the same muscle from appearing multiple times when several exercises target it.

## 8. Python: List Comprehensions

`cur.fetchall()` returns a list of tuples — even a single-column query returns rows like `('Bench Press',)`. A list comprehension is a compact way to pull out just the values needed.

```python
exercise_names = [row[0] for row in cur.fetchall()]
```

Read as: "for each row in the results, take the first item, collect them all into a new list." Equivalent long-form:

```python
exercise_names = []
for row in cur.fetchall():
    exercise_names.append(row[0])
```

A filtered version, used to look up an id matching a selected name:
```python
exercise_id = [row[0] for row in exercises_data if row[1] == selected_exercise][0]
```

## 9. Streamlit: Turning a Python Script into a Web App

Streamlit converts specific Python function calls directly into interactive web page elements — no HTML/CSS/JavaScript required.

| Function | What it creates |
|---|---|
| `st.title(...)` / `st.header(...)` | Page heading |
| `st.write(...)` | Plain text/output on the page |
| `st.selectbox(label, options)` | Dropdown menu |
| `st.number_input(...)` | Numeric input field |
| `st.slider(...)` | Draggable slider |
| `st.form(...)` / `st.form_submit_button(...)` | Groups inputs so nothing submits until one button is clicked |
| `st.session_state` | Persistent storage that survives across reruns (Streamlit normally reruns the whole script on every interaction) |

**Run with:**
```
streamlit run app.py
```
(not `python app.py` — Streamlit needs to start its own live web server)

**Key pattern used in this project:** picking a workout type creates or reuses a `session_id`, stored in `st.session_state` so it persists while multiple exercises are logged against that same session — without Streamlit forgetting it between each form submission.

## 10. Grouping Flat SQL Results in Python

SQL returns flat rows; grouping them into a nested structure (e.g. muscle → list of exercises) happens in Python using a dictionary.

```python
muscles_done = {}
for muscle, exercise, sets, reps, weight in muscle_rows:
    if muscle not in muscles_done:
        muscles_done[muscle] = []
    muscles_done[muscle].append(f"{exercise} ({sets}x{reps} @ {weight}kg)")
```

Note the direct tuple unpacking (`for muscle, exercise, sets, reps, weight in ...`) — an alternative to `row[0]`, `row[1]`, etc. when the columns are known in advance.

## 11. Git & GitHub

```
git init                          # turn a folder into a tracked Git project
git add .                         # stage all changed files
git commit -m "message"           # save a snapshot
git remote add origin <url>       # link to a GitHub repository
git branch -M main
git push -u origin main           # upload
```

**Security practice:** secrets (like `DB_PASSWORD`) are kept in a `.env` file and excluded from Git via `.gitignore`, so they never appear in the public repository or its history.

## Next steps / ideas for later

- [ ] Improve Streamlit form styling/layout
- [ ] Add charts (percentage of Push/Pull/Legs done, muscles trained over time) — likely with Plotly
- [ ] Phase 2: import the larger external exercise reference dataset (Title/Desc/Type/Body Part/Equipment/Level/Rating) and cross-reference against this schema
- [ ] Explore rebuilding the same analysis in Power BI or Tableau as a second, comparative portfolio project

In [ ]:
cur.close()
conn.close()